In [3]:
import pandas as pd

# Pfade zu den Dateien
aapl_path = "data/AAPL_stock_data.csv"
btc_path = "data/BTC-USD_stock_data.csv"
output_path = "data/BTC-USD_stock_data_matched.csv"

# Dateien einlesen
aapl_df = pd.read_csv(aapl_path)
btc_df = pd.read_csv(btc_path)

# Datum parsen
aapl_df["Date"] = pd.to_datetime(aapl_df["Date"], errors='coerce')
btc_df["Date"] = pd.to_datetime(btc_df["Date"], errors='coerce')

# Ungültige Datumswerte entfernen
aapl_df.dropna(subset=["Date"], inplace=True)
btc_df.dropna(subset=["Date"], inplace=True)

# Nur BTC-Zeilen mit Datum in AAPL behalten
common_dates = set(aapl_df["Date"])
btc_df_filtered = btc_df[btc_df["Date"].isin(common_dates)]

# Optional: sortieren & reset index
btc_df_filtered.sort_values("Date", inplace=True)
btc_df_filtered.reset_index(drop=True, inplace=True)

# Speichern
btc_df_filtered.to_csv(output_path, index=False)

print(f"Gefilterte BTC-Daten gespeichert in '{output_path}' ({len(btc_df_filtered)} Zeilen).")

Gefilterte BTC-Daten gespeichert in 'data/BTC-USD_stock_data_matched.csv' (145 Zeilen).


/var/folders/xv/bdrj7gc14wn6s_p3yh4xvvdw0000gn/T/ipykernel_49040/2991950366.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  btc_df_filtered.sort_values("Date", inplace=True)


In [4]:
import os
import pandas as pd
import numpy as np

# Konfiguration
DATA_DIR = "data"  # Ordner mit CSV-Dateien
DATE_COLUMN = "Date"  # Name der Spalte mit dem Datum

def clean_csv(filepath):
    df = pd.read_csv(filepath)

    # Entferne komplett leere Zeilen
    df.dropna(how='all', inplace=True)

    # Entferne Zeilen ohne Datum oder mit ungültigem Datum
    df.dropna(subset=[DATE_COLUMN], inplace=True)
    df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN], errors='coerce')
    df.dropna(subset=[DATE_COLUMN], inplace=True)

    # Identifiziere Kurs-/Volumen-Spalten (alle numerischen außer 'Date')
    data_columns = df.select_dtypes(include=[np.number]).columns.tolist()

    # Entferne Zeilen, bei denen alle Daten-Spalten leer sind
    df.dropna(subset=data_columns, how='all', inplace=True)

    # Runde alle numerischen Spalten auf 2 Nachkommastellen
    df[data_columns] = df[data_columns].round(2)

    # Sortiere nach Datum
    df.sort_values(by=DATE_COLUMN, inplace=True)
    df.reset_index(drop=True, inplace=True)

    return df

def main():
    all_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.csv')]
    print(f"{len(all_files)} CSV-Dateien gefunden.")

    dfs = {}
    date_sets = {}

    for file in all_files:
        filepath = os.path.join(DATA_DIR, file)
        df = clean_csv(filepath)
        dfs[file] = df
        date_sets[file] = set(df[DATE_COLUMN])

        # Überschreibe die gereinigte CSV (optional)
        df.to_csv(filepath, index=False)
        print(f"{file} erfolgreich bereinigt.")

    # Vergleich der Datumseinträge
    reference_dates = None
    mismatches = []

    for file, dates in date_sets.items():
        if reference_dates is None:
            reference_dates = dates
            continue
        if dates != reference_dates:
            mismatches.append(file)

    if mismatches:
        print("\n⚠️ Dateien mit abweichenden Datumswerten gefunden:")
        for file in mismatches:
            diff = reference_dates.symmetric_difference(date_sets[file])
            print(f"- {file} unterscheidet sich an {len(diff)} Datenpunkten.")
    else:
        print("\n✅ Alle Dateien enthalten exakt die gleichen Zeitstempel.")

if __name__ == "__main__":
    main()

12 CSV-Dateien gefunden.
GOOGL_stock_data.csv erfolgreich bereinigt.
AAPL_stock_data.csv erfolgreich bereinigt.
TSLA_stock_data.csv erfolgreich bereinigt.
PLTR_stock_data.csv erfolgreich bereinigt.
META_stock_data.csv erfolgreich bereinigt.
NVDA_stock_data.csv erfolgreich bereinigt.
MSFT_stock_data.csv erfolgreich bereinigt.
GC=F_stock_data.csv erfolgreich bereinigt.
HOOD_stock_data.csv erfolgreich bereinigt.
AMZN_stock_data.csv erfolgreich bereinigt.
BTC-USD_stock_data_matched.csv erfolgreich bereinigt.
UBER_stock_data.csv erfolgreich bereinigt.

✅ Alle Dateien enthalten exakt die gleichen Zeitstempel.


### Single plots percentage changes

In [5]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# Konfiguration
DATA_DIR = "data"
COLUMNS_TO_PLOT = ["Open", "High", "Low", "Volume"]
DATE_COLUMN = "Date"
PLOT_OUTPUT_DIR = "plots"

# Ausgabeordner erstellen, falls nicht vorhanden
os.makedirs(PLOT_OUTPUT_DIR, exist_ok=True)

def compute_percentage_change(df, column):
    return df[column].pct_change() * 100

def process_and_plot_file(filepath):
    filename = os.path.basename(filepath)
    df = pd.read_csv(filepath)

    # Datum parsen
    df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN], errors="coerce")
    df.dropna(subset=[DATE_COLUMN], inplace=True)
    df.sort_values(DATE_COLUMN, inplace=True)

    # Für jede Spalte einen Plot erzeugen
    for col in COLUMNS_TO_PLOT:
        if col not in df.columns:
            print(f"⚠️ Spalte '{col}' nicht in Datei {filename} gefunden. Überspringe.")
            continue

        df[f"{col}_change"] = compute_percentage_change(df, col)

        plt.figure(figsize=(10, 4))
        plt.plot(df[DATE_COLUMN], df[f"{col}_change"], label=f"% Veränderung {col}", color="tab:blue")
        plt.xlabel("Datum")
        plt.ylabel("% Veränderung")
        plt.title(f"{filename}: Tagesveränderung {col} (%)")
        plt.grid(True)
        plt.tight_layout()

        # Dateiname für Plot
        plot_filename = f"{filename.replace('.csv', '')}_{col}_pct_change.png"
        plt.savefig(os.path.join(PLOT_OUTPUT_DIR, plot_filename))
        plt.close()
        print(f"✅ Plot gespeichert: {plot_filename}")

def main():
    csv_files = [f for f in os.listdir(DATA_DIR) if f.endswith(".csv")]
    print(f"{len(csv_files)} CSV-Dateien gefunden.")

    for file in csv_files:
        filepath = os.path.join(DATA_DIR, file)
        process_and_plot_file(filepath)

if __name__ == "__main__":
    main()

12 CSV-Dateien gefunden.
✅ Plot gespeichert: GOOGL_stock_data_Open_pct_change.png
✅ Plot gespeichert: GOOGL_stock_data_High_pct_change.png
✅ Plot gespeichert: GOOGL_stock_data_Low_pct_change.png
✅ Plot gespeichert: GOOGL_stock_data_Volume_pct_change.png
✅ Plot gespeichert: AAPL_stock_data_Open_pct_change.png
✅ Plot gespeichert: AAPL_stock_data_High_pct_change.png
✅ Plot gespeichert: AAPL_stock_data_Low_pct_change.png
✅ Plot gespeichert: AAPL_stock_data_Volume_pct_change.png
✅ Plot gespeichert: TSLA_stock_data_Open_pct_change.png
✅ Plot gespeichert: TSLA_stock_data_High_pct_change.png
✅ Plot gespeichert: TSLA_stock_data_Low_pct_change.png
✅ Plot gespeichert: TSLA_stock_data_Volume_pct_change.png
✅ Plot gespeichert: PLTR_stock_data_Open_pct_change.png
✅ Plot gespeichert: PLTR_stock_data_High_pct_change.png
✅ Plot gespeichert: PLTR_stock_data_Low_pct_change.png
✅ Plot gespeichert: PLTR_stock_data_Volume_pct_change.png
✅ Plot gespeichert: META_stock_data_Open_pct_change.png
✅ Plot gespeich

### Plot: Prozentuale Veränderung Open, Low, High, Volume

In [6]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# Konfiguration
DATA_DIR = "data"
PLOT_OUTPUT_DIR = "plots"
DATE_COLUMN = "Date"
COLUMNS_TO_PLOT = ["Open", "High", "Low", "Volume"]

# Erstelle Ausgabeordner
os.makedirs(PLOT_OUTPUT_DIR, exist_ok=True)

def compute_pct_change(df, column):
    return df[column].pct_change() * 100

def load_and_prepare_csv(filepath):
    df = pd.read_csv(filepath)
    df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN], errors="coerce")
    df.dropna(subset=[DATE_COLUMN], inplace=True)
    df.sort_values(DATE_COLUMN, inplace=True)
    return df

def main():
    csv_files = [f for f in os.listdir(DATA_DIR) if f.endswith(".csv")]
    print(f"{len(csv_files)} CSV-Dateien gefunden.")

    # Dictionary: key = Spaltenname, value = Liste von (DateSeries, ValueSeries, Label)
    plot_data = {col: [] for col in COLUMNS_TO_PLOT}

    for file in csv_files:
        filepath = os.path.join(DATA_DIR, file)
        df = load_and_prepare_csv(filepath)

        for col in COLUMNS_TO_PLOT:
            if col not in df.columns:
                print(f"⚠️ {col} nicht in {file} gefunden, wird übersprungen.")
                continue

            pct_change = compute_pct_change(df, col)
            plot_data[col].append((df[DATE_COLUMN], pct_change, file.replace(".csv", "")))

    # Erzeuge einen Plot pro Spalte
    for col in COLUMNS_TO_PLOT:
        if not plot_data[col]:
            continue  # nichts zu plotten

        plt.figure(figsize=(12, 5))
        for dates, changes, label in plot_data[col]:
            plt.plot(dates, changes, label=label)

        plt.title(f"Prozentuale Veränderung ({col}) zum Vortag")
        plt.xlabel("Datum")
        plt.ylabel("Veränderung in %")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        output_file = os.path.join(PLOT_OUTPUT_DIR, f"{col}_pct_change_comparison.png")
        plt.savefig(output_file)
        plt.close()
        print(f"✅ Plot gespeichert: {output_file}")

if __name__ == "__main__":
    main()

12 CSV-Dateien gefunden.
✅ Plot gespeichert: plots/Open_pct_change_comparison.png
✅ Plot gespeichert: plots/High_pct_change_comparison.png
✅ Plot gespeichert: plots/Low_pct_change_comparison.png
✅ Plot gespeichert: plots/Volume_pct_change_comparison.png


### Plot daily percentage change per stock

In [7]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# Konfiguration
DATA_DIR = "data"
PLOT_OUTPUT_DIR = "plots_per_stock"
DATE_COLUMN = "Date"
COLUMNS_TO_PLOT = ["Open", "High", "Low", "Volume"]

# Ordner für Plots anlegen
os.makedirs(PLOT_OUTPUT_DIR, exist_ok=True)

def compute_pct_change(df, column):
    return df[column].pct_change() * 100

def process_file(filepath):
    filename = os.path.basename(filepath).replace(".csv", "")
    df = pd.read_csv(filepath)

    # Datum vorbereiten
    df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN], errors="coerce")
    df.dropna(subset=[DATE_COLUMN], inplace=True)
    df.sort_values(DATE_COLUMN, inplace=True)

    # Prozentuale Veränderungen berechnen
    available_columns = [col for col in COLUMNS_TO_PLOT if col in df.columns]
    if not available_columns:
        print(f"⚠️ Keine der Zielspalten in {filename} gefunden.")
        return

    plt.figure(figsize=(12, 5))

    for col in available_columns:
        df[f"{col}_change"] = compute_pct_change(df, col)
        plt.plot(df[DATE_COLUMN], df[f"{col}_change"], label=col)

    plt.title(f"{filename} – % Veränderung zum Vortag")
    plt.xlabel("Datum")
    plt.ylabel("Veränderung in %")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    # Speichern
    output_path = os.path.join(PLOT_OUTPUT_DIR, f"{filename}_pct_change_all.png")
    plt.savefig(output_path)
    plt.close()
    print(f"✅ Plot gespeichert: {output_path}")

def main():
    csv_files = [f for f in os.listdir(DATA_DIR) if f.endswith(".csv")]
    print(f"{len(csv_files)} CSV-Dateien gefunden.")

    for file in csv_files:
        process_file(os.path.join(DATA_DIR, file))

if __name__ == "__main__":
    main()

12 CSV-Dateien gefunden.
✅ Plot gespeichert: plots_per_stock/GOOGL_stock_data_pct_change_all.png
✅ Plot gespeichert: plots_per_stock/AAPL_stock_data_pct_change_all.png
✅ Plot gespeichert: plots_per_stock/TSLA_stock_data_pct_change_all.png
✅ Plot gespeichert: plots_per_stock/PLTR_stock_data_pct_change_all.png
✅ Plot gespeichert: plots_per_stock/META_stock_data_pct_change_all.png
✅ Plot gespeichert: plots_per_stock/NVDA_stock_data_pct_change_all.png
✅ Plot gespeichert: plots_per_stock/MSFT_stock_data_pct_change_all.png
✅ Plot gespeichert: plots_per_stock/GC=F_stock_data_pct_change_all.png
✅ Plot gespeichert: plots_per_stock/HOOD_stock_data_pct_change_all.png
✅ Plot gespeichert: plots_per_stock/AMZN_stock_data_pct_change_all.png
✅ Plot gespeichert: plots_per_stock/BTC-USD_stock_data_matched_pct_change_all.png
✅ Plot gespeichert: plots_per_stock/UBER_stock_data_pct_change_all.png


### Plot Raw Values per stock

In [8]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# Konfiguration
DATA_DIR = "data"
PLOT_OUTPUT_DIR = "plots_per_stock"
DATE_COLUMN = "Date"
PRICE_COLUMNS = ["Open", "High", "Low"]
VOLUME_COLUMN = "Volume"

# Ordner für Plots anlegen
os.makedirs(PLOT_OUTPUT_DIR, exist_ok=True)

def process_file(filepath):
    filename = os.path.basename(filepath).replace(".csv", "")
    df = pd.read_csv(filepath)

    # Datum vorbereiten
    df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN], errors="coerce")
    df.dropna(subset=[DATE_COLUMN], inplace=True)
    df.sort_values(DATE_COLUMN, inplace=True)

    # Check, ob nötige Spalten vorhanden sind
    if not all(col in df.columns for col in PRICE_COLUMNS + [VOLUME_COLUMN]):
        print(f"⚠️ Datei {filename} enthält nicht alle benötigten Spalten.")
        return

    fig, ax1 = plt.subplots(figsize=(12, 6))

    # Preisachsen (Open, High, Low)
    for col in PRICE_COLUMNS:
        ax1.plot(df[DATE_COLUMN], df[col], label=col)

    ax1.set_xlabel("Datum")
    ax1.set_ylabel("Preis ($)")
    ax1.legend(loc="upper left")
    ax1.grid(True)

    # Volumen auf zweiter Y-Achse
    ax2 = ax1.twinx()
    ax2.plot(df[DATE_COLUMN], df[VOLUME_COLUMN], label="Volume", color="gray", alpha=0.4)
    ax2.set_ylabel("Volumen (Stück)")
    ax2.legend(loc="upper right")

    plt.title(f"{filename}: Open / High / Low / Volume")
    plt.tight_layout()

    # Speichern
    output_path = os.path.join(PLOT_OUTPUT_DIR, f"{filename}_raw_values.png")
    plt.savefig(output_path)
    plt.close()
    print(f"✅ Plot gespeichert: {output_path}")

def main():
    csv_files = [f for f in os.listdir(DATA_DIR) if f.endswith(".csv")]
    print(f"{len(csv_files)} CSV-Dateien gefunden.")

    for file in csv_files:
        process_file(os.path.join(DATA_DIR, file))

if __name__ == "__main__":
    main()

12 CSV-Dateien gefunden.
✅ Plot gespeichert: plots_per_stock/GOOGL_stock_data_raw_values.png
✅ Plot gespeichert: plots_per_stock/AAPL_stock_data_raw_values.png
✅ Plot gespeichert: plots_per_stock/TSLA_stock_data_raw_values.png
✅ Plot gespeichert: plots_per_stock/PLTR_stock_data_raw_values.png
✅ Plot gespeichert: plots_per_stock/META_stock_data_raw_values.png
✅ Plot gespeichert: plots_per_stock/NVDA_stock_data_raw_values.png
✅ Plot gespeichert: plots_per_stock/MSFT_stock_data_raw_values.png
✅ Plot gespeichert: plots_per_stock/GC=F_stock_data_raw_values.png
✅ Plot gespeichert: plots_per_stock/HOOD_stock_data_raw_values.png
✅ Plot gespeichert: plots_per_stock/AMZN_stock_data_raw_values.png
✅ Plot gespeichert: plots_per_stock/BTC-USD_stock_data_matched_raw_values.png
✅ Plot gespeichert: plots_per_stock/UBER_stock_data_raw_values.png


In [10]:
import yfinance as yf
aapl = yf.Ticker("AAPL")
print(aapl.calendar)   # z.B. Earnings-Termine

{'Dividend Date': datetime.date(2025, 5, 15), 'Ex-Dividend Date': datetime.date(2025, 5, 12), 'Earnings Date': [datetime.date(2025, 7, 30), datetime.date(2025, 8, 4)], 'Earnings High': 1.51, 'Earnings Low': 1.34, 'Earnings Average': 1.42517, 'Revenue High': 90627800000, 'Revenue Low': 86919000000, 'Revenue Average': 88686445280}


### Finnhub for importing company relevent news

In [1]:
import finnhub

# RICHTIG: API-Client mit configuration initialisieren
api_key = "d0q25ehr01qmj4nhjm3gd0q25ehr01qmj4nhjm40"
configuration = finnhub.Configuration()
configuration.api_key['token'] = api_key

client = finnhub.DefaultApi(finnhub.ApiClient(configuration))

# Nachrichten abrufen
news = client.company_news('AAPL', _from='2024-01-01', to='2024-01-31')

# Erste News ausgeben
for n in news[:3]:
    print(f"{n.datetime}: {n.headline} ({n.source})")

AttributeError: module 'finnhub' has no attribute 'Configuration'

In [2]:
import os
print(os.getcwd())      # zeigt den aktuellen Arbeitsordner
print(os.listdir())     # zeigt Dateien in diesem Ordner


/Users/lars/its_labor/Data-science-Project
['cleanData.ipynb', 'Documentation', 'Assignment1', '.DS_Store', 'Stock Market Stock Trend Prediction System.docx', 'import_data', 'README.md', '_old', '.git', 'Change Log', 'data', 'plots']
